# 04. Tổng hợp code đánh giá bộ dữ liệu

Notebook này gom các đoạn code đã dùng để đánh giá bộ dữ liệu, theo thứ tự chạy từ trên xuống:

| Phần | Nội dung | Cần GPU? |
|---|---|---|
| 0 | Tải mã từ GitHub, cài đặt môi trường | không |
| 1 | Chuẩn bị dữ liệu (kèm tự sửa nếu bản chép từ Input bị hỏng), ghi môi trường | không |
| 2 | Xem mã nguồn các hàm `chuan_bi_du_lieu`, `ghi_moi_truong` và kết quả chúng tạo ra | không |
| 3 | Kiểm tra cấu trúc thư mục Input | không |
| 4 | Kiểm tra yaml của tác giả, nhãn thực tế, đối chiếu ảnh, nhãn, map | không |
| 5 | Đo rò rỉ dữ liệu giữa train, val, test (tên ảnh gốc và dHash) | không |
| 6 | Gói kết quả để tải về | không |

**Thiết lập phiên:** Accelerator = None (chạy CPU là đủ), Internet = On, Add Input = `seedlingstemrow`.

## 0. Tải mã nguồn và cài đặt môi trường

In [ ]:
# Bước 1: tải mã nguồn của bạn từ GitHub (repo phải để chế độ Public)
import os, sys, subprocess
GITHUB_USER = "Klark352"
TEN_REPO = "ssr-btl-thigiacmay"
DUONG_DAN = f"/kaggle/working/{TEN_REPO}"
if not os.path.isdir(DUONG_DAN):
    subprocess.run(["git", "clone", "--depth", "1", f"https://github.com/{GITHUB_USER}/{TEN_REPO}.git", DUONG_DAN], check=True)
else:
    subprocess.run(["git", "-C", DUONG_DAN, "pull"], check=True)   # lấy bản mới nhất sau mỗi lần bạn push
sys.path.insert(0, f"{DUONG_DAN}/src")
print("Mã nguồn sẵn sàng:", sorted(os.listdir(f"{DUONG_DAN}/src")))

In [ ]:
# Bước 2: tải repo gốc YOLO-SSR và vá các chỗ không tương thích (an toàn khi chạy lại)
import setup_env, ssr_lib as L
setup_env.cai_dat()

## 1. Chuẩn bị dữ liệu

`L.chuan_bi_du_lieu()` thử lần lượt: dùng thư mục đã có, chép từ bộ đã thu nhỏ trong Input, hoặc thu nhỏ từ dữ liệu gốc.
Nếu bản chép từ Input bị hỏng (thiếu ảnh ở một tập) thì cell dưới xóa nó và thu nhỏ lại từ dữ liệu gốc bằng `chuan_bi_du_lieu(None)`.

In [ ]:
import glob, shutil

def so_anh():
    return {t: len(glob.glob(f"{L.DATA}/images/{t}/*.*")) for t in ("train", "val", "test")}

try:
    L.chuan_bi_du_lieu()
except RuntimeError as loi:
    print("CẢNH BÁO:", loi)
print("Số ảnh sau lần chuẩn bị đầu:", so_anh())
if not all(so_anh().values()):
    print("Thiếu ảnh ở một tập: xóa bản chép hỏng và thu nhỏ lại từ dữ liệu gốc (chậm).")
    shutil.rmtree(L.DATA, ignore_errors=True)
    L.chuan_bi_du_lieu(None)
print("Số ảnh cuối cùng:", so_anh())
L.ghi_moi_truong()

## 2. Xem mã nguồn các hàm và kết quả chúng tạo ra

Dùng để trình bày với giáo viên: hai hàm gồm những bước nào và sinh ra file gì.

In [ ]:
import inspect

def xem_ham(*cac_ham):
    for h in cac_ham:
        dong, dau = inspect.getsourcelines(h)
        print("=" * 70)
        print(f"{h.__name__}  (ssr_lib.py, dòng {dau})")
        print("=" * 70)
        print("".join(f"{dau + i:4d}  {s}" for i, s in enumerate(dong)))

def lay(*ten):
    return [getattr(L, t) for t in ten if hasattr(L, t)]

# Nhóm 1: chuẩn bị dữ liệu
xem_ham(*lay("chuan_bi_du_lieu", "tim_nguon_da_thu_nho", "thu_nho_du_lieu", "_xu_ly_mot_tep",
             "_thu_muc_anh", "_du_ba_tap", "tao_yaml_du_lieu", "_id_lop_trong_nhan"))

In [ ]:
# Nhóm 2: ghi môi trường
xem_ham(*lay("ghi_moi_truong", "_chay"))

In [ ]:
# Kết quả hai hàm tạo ra: file yaml dữ liệu và đầu file nhật ký môi trường
print(open(L.YAML_DU_LIEU, encoding="utf-8").read())
print("-" * 70)
nhat_ky = open(f"{L.LOGS}/moi_truong.txt", encoding="utf-8").read().splitlines()
print("\n".join(nhat_ky[:6]), f"\n... (còn {len(nhat_ky) - 6} dòng pip freeze)")

## 3. Kiểm tra cấu trúc thư mục Input

In [ ]:
# Liệt kê các thư mục trong Dataset gốc để biết có những bộ dữ liệu nào (ví dụ SSR và SSR_expansion_1)
import os
goc = os.path.dirname(os.path.dirname(L.NGUON_GOC))      # thư mục gốc của Dataset trong Input
print("Dataset:", goc)
if not os.path.isdir(goc):
    print("Không thấy thư mục này, kiểm tra lại mục Add Input.")
for d, ds, tep in os.walk(goc):
    cap = d[len(goc):].count(os.sep)
    if cap > 4:
        ds[:] = []
        continue
    ds.sort()
    anh = sum(f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp")) for f in tep)
    txt = sum(f.endswith(".txt") for f in tep)
    print("  " * cap + os.path.basename(d) + f"/  (ảnh: {anh}, txt: {txt})")

## 4. Kiểm tra yaml của tác giả, nhãn thực tế và đối chiếu ảnh, nhãn, map

In [ ]:
# Các file yaml trong repo gốc có khóa labels_list (cấu hình đa nhiệm của tác giả)
import glob, os
R = L.REPO
for f in sorted(glob.glob(f"{R}/**/*.yaml", recursive=True)):
    t = open(f, encoding="utf-8", errors="ignore").read()
    if "labels_list" in t:
        print("#", os.path.relpath(f, R))
        print(t[:1800])
        print("-" * 40)

In [ ]:
# Nhãn thực tế của từng nhiệm vụ: số file, số dòng, id lớp, số cột mỗi dòng (cho biết nhãn là hộp hay đa giác)
import collections, glob, os
for nv in ("detection-object", "seg-line"):
    for tap in ("train", "val", "test"):
        tep = sorted(glob.glob(f"{L.DATA}/{nv}/labels/{tap}/*.txt"))
        lop, so_dong, cot = collections.Counter(), 0, collections.Counter()
        for f in tep:
            for dong in open(f, encoding="utf-8").read().splitlines():
                if dong.strip():
                    cac = dong.split()
                    lop[cac[0]] += 1
                    cot[len(cac)] += 1
                    so_dong += 1
        print(f"{nv:<18} {tap:<5}: {len(tep):4} file nhãn | {so_dong:6} dòng | id lớp: {dict(lop)} | "
              f"số cột mỗi dòng (phổ biến nhất): {cot.most_common(3)}")
    mau = sorted(glob.glob(f"{L.DATA}/{nv}/labels/train/*.txt"))
    if mau:
        dong_dau = open(mau[0], encoding="utf-8").read().splitlines()[:1]
        print("  ví dụ:", os.path.basename(mau[0]), "->", (dong_dau[0][:90] if dong_dau else "(file rỗng)"))
    else:
        print("  không có file nhãn train cho", nv)

In [ ]:
# Tạo lại yaml dữ liệu và đối chiếu số ảnh, số file nhãn, id lớp với map (cùng phần kiểm tra trong chuan_bi_du_lieu)
L.tao_yaml_du_lieu()

## 5. Đo rò rỉ dữ liệu giữa train, val, test

Ba phép đo, từ thô đến kỹ:
1. `ro_ri.phan_tich()`: trùng tên ảnh gốc và khoảng cách dHash 64 bit tới ảnh train gần nhất.
2. Liệt kê các cặp cùng tên ảnh gốc kèm khoảng cách dHash, để biết là ảnh chụp liên tiếp hay chỉ trùng tên.
3. Thử cả 8 phép lật và xoay, rồi hiển thị 6 cặp giống nhất để nhìn bằng mắt.

Ngưỡng: khoảng cách ≤ 5 là gần như trùng, ≤ 10 là rất giống (tối đa 64).

In [ ]:
import ro_ri
ket_qua = ro_ri.phan_tich()

In [ ]:
# Các ảnh val/test cùng tên ảnh gốc với một ảnh train: đo khoảng cách dHash của từng cặp
import glob, os
from ro_ri import anh_goc, dhash, ANH, DATA

tep = {t: sorted(f for f in glob.glob(f"{DATA}/images/{t}/*") if f.lower().endswith(ANH))
       for t in ("train", "val", "test")}
theo_goc = {}
for f in tep["train"]:
    theo_goc.setdefault(anh_goc(f), []).append(f)

for t in ("val", "test"):
    print("==", t)
    for f in tep[t]:
        g = anh_goc(f)
        for tr in theo_goc.get(g, []):
            kc = int((dhash(f) != dhash(tr)).sum())
            print(f"  {os.path.basename(f)[:42]}  <->  {os.path.basename(tr)[:42]}  khoảng cách {kc}")

In [ ]:
# Thử cả 8 phép lật/xoay để bắt ảnh trùng bị xoay hoặc lật, rồi hiển thị 6 cặp giống nhất
import glob, os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from ro_ri import ANH, DATA

tep = {t: sorted(f for f in glob.glob(f"{DATA}/images/{t}/*") if f.lower().endswith(ANH))
       for t in ("train", "val", "test")}

def hash_bien_the(duong_dan, day_du=True):
    """dHash của ảnh nguyên bản và 7 phép lật/xoay còn lại. Hàng 0 là nguyên bản."""
    with Image.open(duong_dan) as a:
        a = a.convert("L"); a.load()
    bt = []
    for xoay in (0, 90, 180, 270):
        b = a.rotate(xoay, expand=True)
        for lat in (False, True):
            c = ImageOps.mirror(b) if lat else b
            p = np.asarray(c.resize((9, 8), Image.LANCZOS), dtype=np.int16)
            bt.append((p[:, 1:] > p[:, :-1]).flatten())
            if not day_du:
                return np.array(bt)
    return np.array(bt)

train = np.array([hash_bien_the(f, day_du=False)[0] for f in tep["train"]])
ket = []
for t in ("val", "test"):
    for f in tep[t]:
        bt = hash_bien_the(f)                                # (8, 64)
        kc = (bt[:, None, :] != train[None, :, :]).sum(-1)   # (8, số ảnh train)
        k, j = np.unravel_index(kc.argmin(), kc.shape)
        ket.append((int(kc[k, j]), t, f, tep["train"][j], int(k)))
ket.sort()

for t in ("val", "test"):
    d = [x[0] for x in ket if x[1] == t]
    print(f"{t}: <=5: {sum(x <= 5 for x in d)}  <=10: {sum(x <= 10 for x in d)}  (trên {len(d)} ảnh)")
print("10 cặp giống nhất sau khi thử lật/xoay (biến thể 0 = nguyên bản):")
for kc, t, f, tr, k in ket[:10]:
    print(f"  {kc:2d} {t:4s} {os.path.basename(f)[:30]} <-> {os.path.basename(tr)[:30]}  biến thể {k}")

fig, ax = plt.subplots(2, 6, figsize=(24, 9))
for i, (kc, t, f, tr, k) in enumerate(ket[:6]):
    r, c = divmod(i, 3)
    ax[r][c * 2].imshow(Image.open(f));      ax[r][c * 2].set_title(f"{t} (khoảng cách {kc})")
    ax[r][c * 2 + 1].imshow(Image.open(tr)); ax[r][c * 2 + 1].set_title("train gần nhất")
for a in ax.ravel():
    a.axis("off")
plt.tight_layout()
os.makedirs(L.LOGS, exist_ok=True)
plt.savefig(f"{L.LOGS}/cap_giong_nhat.png", dpi=110, bbox_inches="tight")
plt.show()

## 6. Gói kết quả để tải về

Nén các file kết quả vào `bao_cao_du_lieu.zip` (mở panel Output bên phải, tải file này về).

**Kết quả đo ở lần chạy trước, để đối chiếu** (845 / 179 / 183 ảnh ở train / val / test):

| Phép đo | val | test |
|---|---|---|
| Cùng tên ảnh gốc với một ảnh train | 9 (5,0%) | 8 (4,4%) |
| dHash ≤ 5, gần như trùng | 0 | 0 |
| dHash ≤ 10 (nguyên bản) | 5 (2,8%) | 3 (1,6%) |
| dHash ≤ 10 (thử 8 phép lật/xoay) | 9 (5,0%) | 6 (3,3%) |

Ba ảnh gốc xuất hiện ở cả val và test. Nhìn bằng mắt, có ít nhất hai cặp ở test là ảnh chụp liên tiếp cùng cảnh.

In [ ]:
import zipfile, os
dich = f"{L.BASE}/bao_cao_du_lieu.zip"
tep_can = [L.YAML_DU_LIEU, f"{L.LOGS}/moi_truong.txt", f"{L.LOGS}/ro_ri.json", f"{L.LOGS}/cap_giong_nhat.png"]
with zipfile.ZipFile(dich, "w", zipfile.ZIP_DEFLATED) as z:
    for f in tep_can:
        if os.path.exists(f):
            z.write(f, os.path.basename(f))
        else:
            print("Không có", f)
print("Đã nén:", dich, f"({os.path.getsize(dich) / 1e3:.0f} KB)")